# HybridBayesTree

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridBayesTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam

A `HybridBayesTree` is the hybrid equivalent of a `gtsam.GaussianBayesTree`. It represents the result of **multifrontal** variable elimination on a `HybridGaussianFactorGraph`.

Like a standard Bayes tree, it's a tree structure where each node is a 'clique'. However, in a `HybridBayesTree`, each clique contains a `gtsam.HybridConditional` representing $P(F_k | S_k)$, where $F_k$ are the frontal variables (both discrete and continuous) eliminated in that clique, and $S_k$ are the separator variables (again, potentially mixed discrete and continuous) shared with the parent clique.

The joint probability distribution $P(X, M)$ encoded by the tree is the product of all clique conditionals:
$$
P(X, M) = \prod_k P(F_k | S_k)
$$
Compared to a `HybridBayesNet` (from sequential elimination), a `HybridBayesTree` often has a shallower structure, which can be advantageous for inference tasks like marginalization and incremental updates, especially in sparse problems common in robotics (SLAM).

In [ ]:
import gtsam
import numpy as np
from gtsam import symbol_shorthand

# Graphs & Elimination
from gtsam import HybridGaussianFactorGraph, Ordering, HybridBayesTree
from gtsam import JacobianFactor, DecisionTreeFactor, DiscreteKey, HybridGaussianFactor

# Values
from gtsam import DiscreteValues, VectorValues, HybridValues

# Results
from gtsam import GaussianBayesTree

X = symbol_shorthand.X # Continuous Key
D = symbol_shorthand.D # Discrete Key

## Creating a HybridBayesTree (via Elimination)

`HybridBayesTree` objects are obtained by performing multifrontal elimination on a `HybridGaussianFactorGraph`.

In [ ]:
# --- Create a HybridGaussianFactorGraph ---
hgfg = gtsam.HybridGaussianFactorGraph()
dk0 = DiscreteKey(D(0), 2) # Binary discrete variable

# Prior on D0: P(D0=0)=0.6, P(D0=1)=0.4
prior_d0 = DecisionTreeFactor([dk0], "0.6 0.4")
hgfg.add(prior_d0)

# Prior on X0: P(X0) = N(0, 1)
prior_x0 = JacobianFactor(X(0), np.eye(1), np.zeros(1), gtsam.noiseModel.Isotropic.Sigma(1, 1.0))
hgfg.add(prior_x0)

# Conditional measurement on X1: P(X1 | D0)
# Mode 0: P(X1 | D0=0) = N(1, 0.25)
gf0 = JacobianFactor(X(1), np.eye(1), np.array([1.0]), gtsam.noiseModel.Isotropic.Sigma(1, 0.5))
# Mode 1: P(X1 | D0=1) = N(5, 1.0)
gf1 = JacobianFactor(X(1), np.eye(1), np.array([5.0]), gtsam.noiseModel.Isotropic.Sigma(1, 1.0))
meas_x1_d0 = HybridGaussianFactor(dk0, [gf0, gf1])
hgfg.add(meas_x1_d0)

# Factor connecting X0 and X1: P(X1 | X0) = N(X0+1, 0.1)
odom_x0_x1 = JacobianFactor(X(0), -np.eye(1), X(1), np.eye(1), np.array([1.0]), gtsam.noiseModel.Isotropic.Sigma(1, np.sqrt(0.1)))
hgfg.add(odom_x0_x1)

print("Original HybridGaussianFactorGraph:")
hgfg.print()

# --- Elimination ---
# Use default hybrid ordering (continuous first, then discrete)
ordering = gtsam.HybridOrdering(hgfg)
print(f"\nElimination Ordering: {ordering}")

# Perform multifrontal elimination
hybrid_bayes_tree = hgfg.eliminateMultifrontal(ordering)

print("\nResulting HybridBayesTree:")
hybrid_bayes_tree.print() # Might only show root clique if simple

## Operations on HybridBayesTree

Similar to `HybridBayesNet`, the tree can be used for optimization, evaluation, and extracting specific conditional distributions.

In [ ]:
hbt = hybrid_bayes_tree # Use the tree from elimination

# --- Optimization (Finding MAP state) ---
# Computes MPE for discrete, then optimizes continuous given MPE
map_solution = hbt.optimize()
print("\nMAP Solution (Optimize):")
map_solution.print()

# --- MPE (Most Probable Explanation for Discrete Variables) ---
mpe_assignment = hbt.mpe()
print("\nMPE Discrete Assignment:")
mpe_assignment.print()

# --- Optimize Continuous given specific Discrete Assignment ---
cont_solution_d0_eq_0 = hbt.optimize(gtsam.DiscreteValues([(D(0), 0)]))
print("\nOptimized Continuous Solution for D0=0:")
cont_solution_d0_eq_0.print()

cont_solution_d0_eq_1 = hbt.optimize(gtsam.DiscreteValues([(D(0), 1)]))
print("\nOptimized Continuous Solution for D0=1:")
cont_solution_d0_eq_1.print()

# --- Evaluation ---
# Use the MAP solution for evaluation
log_prob = hbt.logProbability(map_solution)
# Note: logProbability on BayesTree/BayesNet sums log conditionals.
# It should match the logProbability of the original graph (up to constants)
# evaluated at the MAP solution, but calculation might differ slightly internally.

# Evaluate error (sum of errors of clique conditionals)
total_error = hbt.error(map_solution)
print(f"\nTotal Error at MAP solution: {total_error}")
print(f"LogProbability at MAP solution: {log_prob}") # logProb = -error - sum(negLogConstant)

# --- Extract Conditional Distributions ---
# Choose a specific GaussianBayesTree for a discrete assignment
gaussian_bayes_tree_d0_eq_0 = hbt.choose(gtsam.DiscreteValues([(D(0), 0)]))
print("\nGaussianBayesTree for D0=0:")
gaussian_bayes_tree_d0_eq_0.print()

# Note: Extracting marginals (like hbt.marginalFactor(key)) might be
# less straightforward than in GaussianBayesTree due to the discrete variables.
# Often, one uses choose() first.

## Visualization

The structure of the Bayes tree (cliques and their parent-child relationships) can be visualized.

In [ ]:
import graphviz

try:
    dot_string = hbt.dot()
    display(graphviz.Source(dot_string))
except Exception as e:
    print(f"Error generating DOT string: {e}")